# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

# =========================
# 1. Connect to internship repo
# =========================

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            "requirements.txt"
        ],
        check=True
    )
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# =========================
# 2. Hugging Face access
# =========================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

print("HF_TOKEN found.")

# =========================
# 3. DuckDB
# =========================

import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face.")

# =========================
# 4. March 2026 warehouse
# =========================

march_df = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        client_has_gsc,
        client_has_ga4,
        gsc_data_available,
        ga4_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        scroll_events,
        sessions_ai,
        sessions_paid,
        sessions_direct,
        sessions_social,
        sessions_organic,
        month
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("Working directory:", os.getcwd())
print("March 2026 shape:", march_df.shape)
print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

display(march_df.head())

HF_TOKEN found.
DuckDB connected to Hugging Face.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Working directory: /content/flyrank-ml-internship-starter
March 2026 shape: (9841378, 17)
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai,sessions_paid,sessions_direct,sessions_social,sessions_organic,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "scroll_events",
    "sessions_organic"
]

feature_frame = march_df[feature_cols].copy()

print("Original shape:", feature_frame.shape)

display(feature_frame.head())

Original shape: (9841378, 5)


,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_organic
0,20,0,67,<NA>,<NA>
1,1,0,0,<NA>,<NA>
2,125,1,616,<NA>,<NA>
3,7,0,28,<NA>,<NA>
4,11,0,25,<NA>,<NA>


In [3]:
# Numeric features: median imputation
for col in feature_cols:
    feature_frame[col] = pd.to_numeric(
        feature_frame[col],
        errors="coerce"
    )

feature_frame = feature_frame.fillna(
    feature_frame.median(numeric_only=True)
)

print("Missing values after fill:")
display(feature_frame.isna().sum())

print("Final feature shape:", feature_frame.shape)

Missing values after fill:


,0
gsc_impressions,0
gsc_clicks,0
gsc_sum_position,0
scroll_events,0
sessions_organic,0


Final feature shape: (9841378, 5)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

# Numeric features: median imputation
for col in feature_cols:
    feature_frame[col] = pd.to_numeric(
        feature_frame[col],
        errors="coerce"
    )

feature_frame = feature_frame.fillna(
    feature_frame.median(numeric_only=True)
)

print("Missing values after fill:")
display(feature_frame.isna().sum())

print("Final feature shape:", feature_frame.shape)

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Clean feature columns:")
print(feature_frame.columns.tolist())

Clean feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'scroll_events', 'sessions_organic']


In [6]:
all_columns = march_df.columns.tolist()

trend_like_columns = [
    col for col in all_columns
    if any(
        word in col.lower()
        for word in ["trend", "delta", "change", "target", "label"]
    )
]

print("Potential label/trend-derived columns:")
print(trend_like_columns)

Potential label/trend-derived columns:
[]


In [8]:
print(
    "No trend/label-derived field was found in the selected March slice, "
    "so no artificial leakage column was added."
)

No trend/label-derived field was found in the selected March slice, so no artificial leakage column was added.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

### Trend or label-derived fields
Excluded because they can directly encode the target/proxy and create leakage.

### Future-window outcomes
Excluded because they would not be known at the moment the decision is made.

### Product or action flags
Excluded because a product-generated flag may already contain the decision logic being evaluated, which would make the feature set circular.

### Client-identifying information
Excluded because the warehouse is pseudonymized and the assignment requires no client-identifying output.

### Unavailable-source indicators as performance features
Availability fields are retained as context rather than treated as performance outcomes. Missingness can represent source availability rather than poor performance.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.